# Phase 9 — Hybrid ML System

This notebook compares three approaches on FraudShield's chronological test block:

1. **XGBoost** supervised fraud score.
2. **PyTorch autoencoder** reconstruction error trained on legitimate transactions only.
3. **Hybrid stacker** combining normalized XGBoost and anomaly signals with regularized Logistic Regression.

The model's threshold is calibrated on legitimate examples in the later validation calibration segment to target approximately 1% validation FPR. The final test block is not used to fit models or thresholds.

> Important: the hybrid score is a ranking score, not a calibrated probability. The dataset has very few fraud cases, and a dedicated probability calibration block would be needed before probabilistic claims.

Run this notebook from the project root: `D:\projects\fraudshield-ai`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports' / 'hybrid'
summary = json.loads((REPORTS / 'hybrid_summary.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(REPORTS / 'test_model_comparison.csv')


## 1. The experimental protocol

The base models are fitted on the first chronological training block. The validation block is separated again: its early 70% fits the hybrid Logistic Regression meta-model, and its later 30% is used to set each score threshold using legitimate examples only. This reduces the temptation to evaluate the hybrid on rows it used to fit itself.

The autoencoder in Phase 9 differs methodologically from Phase 8: it is early-stopped using reconstruction loss on legitimate validation examples only, not validation fraud labels.

In [ ]:
display(pd.DataFrame(summary['partition_counts']).T)
print('Normalization:', summary['normalization'])
print('Threshold policy:', summary['threshold_policy'])


## 2. Test comparison

Threshold-dependent metrics use each model's validation-calibrated threshold targeting approximately 1% false-positive rate on legitimate validation records. Test FPR may differ because the distribution shifts across time.

In [ ]:
display(comparison[['model', 'average_precision', 'roc_auc', 'precision', 'recall', 'f1', 'false_positive', 'false_negative', 'threshold']].round(4))


In [ ]:
display(Image(filename=str(REPORTS / 'test_model_comparison.png')))
display(Image(filename=str(REPORTS / 'test_precision_recall_curves.png')))


## 3. Is the anomaly signal complementary?

We compare fraud cases caught under the same validation target-FPR protocol. A useful secondary model should ideally flag fraud transactions missed by XGBoost without adding too many false alarms.

In [ ]:
display(pd.Series(summary['test_complementarity'], name='count').to_frame())
display(pd.DataFrame(summary['hybrid_coefficients'].items(), columns=['normalized signal', 'coefficient']))


## 4. Inspect individual test scores

`hybrid_score_0_100_not_probability` is a sigmoid transformation of the Logistic Regression decision margin for display only. It is not a calibrated fraud probability.

In [ ]:
predictions = pd.read_csv(REPORTS / 'test_predictions.csv')
display(predictions.head(10))
display(predictions.sort_values('hybrid_margin', ascending=False).head(10))


## 5. Re-run experiment (optional)

This trains the Phase 9 models again and regenerates artifacts. CPU training is usually modest for this architecture; use `--device auto` or run the command below from PowerShell if CUDA is configured.

```powershell
python -m src.models.hybrid_system --device auto --n-jobs 2
```

## Conclusion

Select models based on measured value, not architectural complexity. In this run, the hybrid's recall increases slightly, but its Average Precision is lower than XGBoost. That is not enough evidence to replace XGBoost as the primary score. A new later-time holdout is recommended for a final independent performance claim.